<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

# =============================================================================
# CreditFlow — P2 EDA with Synthetic Data (Colab-ready)
# =============================================================================
#
# Product Spec: docs/spec.md v0.1 (2026-09-08)
# PRD Source: plans/prds/creditflow-p2-p3.md
# Leakage Suspects: plans/prds/creditflow-p1.md §8
#
# DISCLAIMER — PROXY DATASET:
# This notebook uses SYNTHETIC data generated for simulation purposes only.
# It is NOT real banking data. No claim is made about deployment for
# actual financial institutions. All artifacts must clearly state "simulation/proxy".
# See docs/spec.md v0.1 §19 for proxy dataset disclaimer.
#
# Constraints:
#   - NO .fit(), NO model training, NO xgboost/torch/tensorflow imports
#   - NO GPU usage
#   - All code runnable on Colab with requirements-colab.txt
#   - Every print statement includes P(default) where relevant
#   - Synthetic data generated with random_state=42 for reproducibility
# =============================================================================

In [1]:
%pip install -q -r requirements-colab.txtimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings("ignore")# Display settingspd.set_option("display.max_columns", None)pd.set_option("display.max_rows", 100)pd.set_option("display.width", 200)pd.set_option("display.float_format", "{:.4f}".format)plt.rcParams["figure.figsize"] = (10, 6)plt.rcParams["font.size"] = 12# Set random state for reproducibilityrandom_state = 42np.random.seed(random_state)# --- Generate Synthetic Dataset (1000 rows) ---n = 1000income = np.random.lognormal(mean=7.5, sigma=0.5, size=n).astype(int)income = np.clip(income, 1500, 15000)age = np.random.randint(18, 100, size=n)employment_years = np.random.uniform(0, np.maximum(age - 18, 0), size=n)employment_years = np.round(employment_years, 2)loan_amount = np.random.lognormal(mean=9.0, sigma=0.6, size=n).astype(int)loan_amount = np.clip(loan_amount, 5000, 50000)loan_term = np.random.choice([12, 24, 36, 48, 60], size=n)existing_debt = np.random.uniform(0, income * 0.8, size=n)existing_debt = np.round(existing_debt, 2)credit_history = np.random.uniform(1, np.maximum(age - 18, 1), size=n)credit_history = np.round(credit_history, 2)previous_defaults = np.random.poisson(lam=0.3, size=n)default = np.random.binomial(1, p=0.12, size=n)# Build DataFramedf = pd.DataFrame({    'income': income,    'age': age,    'employment_years': employment_years,    'loan_amount': loan_amount,    'loan_term': loan_term,    'existing_debt': existing_debt,    'credit_history': credit_history,    'previous_defaults': previous_defaults,    'default': default,})# Print dataset infoprint("=" * 70)print("SYNTHETIC DATASET GENERATED")print("=" * 70)print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")print(f"Columns: {list(df.columns)}")print("\nFirst 5 rows:")print(df.head().to_string())print(f"\nP(default) globally: {df['default'].mean():.4f}")print(f"Number of defaults: {df['default'].sum()}")print(f"Number of non-defaults: {len(df) - df['default'].sum()}")print("=" * 70)print("Proxy dataset disclaimer: Simulation only. See docs/spec.md v0.1.")print("=" * 70)

zsh: parse error near `)'


Note: you may need to restart the kernel to use updated packages.


# =============================================================================
# Section 1: Synthetic Data Generated
# =============================================================================
#
# The synthetic dataset has been generated with 1000 rows and 9 columns
# (8 raw features + target).
#
# Raw features: income, age, employment_years, loan_amount, loan_term,
#               existing_debt, credit_history, previous_defaults
# Target: default (binary, ~12% default rate)
#
# All constraints from docs/spec.md v0.1 §5 are enforced:
#   - income > 0, loan_amount > 0
#   - age in [18, 100]
#   - employment_years >= 0, <= age - 18
#   - existing_debt >= 0, <= income * 0.8
#   - credit_history >= 1, <= age - 18
#   - previous_defaults >= 0
#   - P(default) ≈ 0.12 (simulating class imbalance)
# =============================================================================

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [2]:
print("\n" + "=" * 70)print("SECTION 2: Q1 — DEFAULT CUSTOMER CHARACTERISTICS")print("=" * 70)# --- Global P(default) ---print(f"\n--- Global P(default) ---")p_default_global = df["default"].mean()print(f"P(default) globally: {p_default_global:.4f}")print(f"Number of defaults: {df['default'].sum()}")print(f"Total applications: {len(df)}")# --- P(default) by Age Groups ---print(f"\n--- P(default) by Age Groups ---")age_bins = [0, 25, 35, 45, 55, 100]age_labels = ["18-25", "26-35", "36-45", "46-55", "56+"]df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=False)age_default_rate = df.groupby("age_group")["default"].agg(["mean", "count"])age_default_rate.columns = ["P(default)", "count"]print(age_default_rate)# --- P(default) by Income Brackets ---print(f"\n--- P(default) by Income Brackets ---")income_bins = [0, 2000, 4000, 6000, 10000, float("inf")]income_labels = ["<2k", "2k-4k", "4k-6k", "6k-10k", "10k+"]df["income_bracket"] = pd.cut(df["income"], bins=income_bins, labels=income_labels, right=False)income_default_rate = df.groupby("income_bracket")["default"].agg(["mean", "count"])income_default_rate.columns = ["P(default)", "count"]print(income_default_rate)# --- P(default) by Employment Years Groups ---print(f"\n--- P(default) by Employment Years Groups ---")emp_bins = [-1, 0, 2, 5, 10, 100]emp_labels = ["0 yrs", "1-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]df["employment_group"] = pd.cut(df["employment_years"], bins=emp_bins, labels=emp_labels, right=False)emp_default_rate = df.groupby("employment_group")["default"].agg(["mean", "count"])emp_default_rate.columns = ["P(default)", "count"]print(emp_default_rate)# --- P(default) by previous_defaults Count ---print(f"\n--- P(default) by previous_defaults Count ---")prev_default_rate = df.groupby("previous_defaults")["default"].agg(["mean", "count"])prev_default_rate.columns = ["P(default)", "count"]print(prev_default_rate)print(f"\nNote: previous_defaults is a leakage suspect (see Section 4).")# --- Visualization: P(default) by Age Group ---print(f"\n--- Visualization: P(default) by Age Group ---")fig, ax = plt.subplots(figsize=(10, 6))age_rates = age_default_rate["P(default)"].valuesage_labels_plot = age_default_rate.index.astype(str)colors = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6"]bars = ax.bar(age_labels_plot, age_rates, color=colors)ax.set_xlabel("Age Group")ax.set_ylabel("P(default)")ax.set_title("P(default) by Age Group (Synthetic Data — Simulation)")ax.set_ylim(0, max(age_rates) * 1.2)for bar, val in zip(bars, age_rates):    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,            f"{val:.3f}", ha="center", va="bottom", fontsize=10)plt.tight_layout()plt.savefig("p_default_by_age.png", dpi=150, bbox_inches="tight")plt.show()print("Chart saved: p_default_by_age.png")print("\nQ1 complete. P(default) computed globally and by key features.")

SyntaxError: invalid syntax (3616350207.py, line 1)

# =============================================================================
# Section 2: Q2 — Feature Correlation with Default
# =============================================================================
#
# Investigate which raw features are most correlated with the target
# variable `default`. This helps identify the strongest predictors
# of credit risk in the synthetic dataset.
#
# Per docs/spec.md v0.1 §6 (Q2):
#   - Correlation matrix for all 8 raw features + target
#   - Print correlation with target, sorted by absolute value
#   - Heatmap visualization
# =============================================================================

In [ ]:
print("\n" + "=" * 70)print("SECTION 2 continued: Q2 — FEATURE CORRELATION WITH DEFAULT")print("=" * 70)# --- Raw Features + Target Correlation ---print(f"\n--- Correlation Matrix (Raw Features + Target) ---")raw_features = ["income", "age", "employment_years", "loan_amount",                "loan_term", "existing_debt", "credit_history",                "previous_defaults"]target = "default"all_cols = raw_features + [target]# Ensure all columns existavailable_cols = [c for c in all_cols if c in df.columns]corr_matrix = df[available_cols].corr()# Print correlation with targetprint(f"\nCorrelation with P(default) (target='default'):")default_corr = corr_matrix[target].drop(target).sort_values(key=abs, ascending=False)print(default_corr)# Highlight top correlated featuresprint(f"\nTop features correlated with P(default):")for feat, corr_val in default_corr.head(5).items():    print(f"  {feat}: {corr_val:.4f}")# --- Visualization: Correlation Heatmap ---print(f"\n--- Correlation Heatmap ---")plt.figure(figsize=(12, 10))sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r",            center=0, square=True, linewidths=0.5)plt.title("Feature Correlation Matrix (Synthetic Data — Simulation)")plt.tight_layout()plt.savefig("correlation_heatmap.png", dpi=150, bbox_inches="tight")plt.show()print("Heatmap saved: correlation_heatmap.png")# --- Derived Features Correlations (if available) ---print(f"\n--- Derived Features Correlations (if available) ---")derived_features = ["debt_to_income", "loan_to_income", "debt_to_loan",                    "employment_stability", "credit_history_year_ratio"]available_derived = [f for f in derived_features if f in df.columns]if available_derived:    derived_corr = df[available_derived + [target]].corr()    print(f"\nCorrelation of derived features with P(default):")    for feat in available_derived:        print(f"  {feat}: {derived_corr.loc[feat, target]:.4f}")else:    print("Derived features not yet available. Run P3 Feature Engineering first.")    print("See pipeline/feature_engineering/features.py for add_derived_features().")print("\nQ2 complete. Feature correlations with P(default) computed.")

# =============================================================================
# Section 3: Q3 — Class Imbalance
# =============================================================================
#
# Check the class distribution of the target variable `default`.
# Per docs/spec.md v0.1 §9:
#   - Class imbalance must be identified (Default ~12% / Non-default ~88%)
#   - Accuracy ~88% is near-meaningless if model always predicts non-default
#   - Prioritize Recall/F1 because cost of False Negative > False Positive
# =============================================================================

In [ ]:
print("\n" + "=" * 70)print("SECTION 3: Q3 — CLASS IMBALANCE")print("=" * 70)# --- Class Distribution ---print(f"\n--- Class Distribution (P(default)) ---")class_counts = df["default"].value_counts()class_pct = df["default"].value_counts(normalize=True) * 100for cls, count in class_counts.items():    p_default = class_pct[cls]    label = "default (1)" if cls == 1 else "non-default (0)"    print(f"P(default)={cls} ({label}): count={count}, percentage={p_default:.2f}%")total = len(df)n_default = class_counts.get(1, 0)n_non_default = class_counts.get(0, 0)if n_non_default > 0:    imbalance_ratio = n_non_default / max(n_default, 1)    print(f"\nImbalance ratio (non-default / default): {imbalance_ratio:.2f}:1")    if imbalance_ratio > 5:        print(f"⚠️  WARNING: Severe class imbalance detected (>5:1).")        print(f"    P(default) minority class is very low.")        print(f"    Consider class_weight or sampling in P4 model training.")    else:        print(f"Imbalance ratio is within acceptable range (≤5:1).")# --- Visualization: Class Imbalance Bar Chart ---print(f"\n--- Class Imbalance Visualization ---")fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Bar chart of countscolors = ["#3498db", "#e74c3c"]axes[0].bar(class_counts.index, class_counts.values, color=colors)axes[0].set_xlabel("Class (0=Non-default, 1=Default)")axes[0].set_ylabel("Count")axes[0].set_title("Class Distribution — Counts")axes[0].set_xticks([0, 1])# Bar chart of percentagesaxes[1].bar(class_pct.index, class_pct.values, color=colors)axes[1].set_xlabel("Class (0=Non-default, 1=Default)")axes[1].set_ylabel("Percentage (%)")axes[1].set_title("Class Distribution — Percentages")axes[1].set_xticks([0, 1])plt.suptitle("Class Imbalance Analysis (Synthetic Data — Simulation)", fontsize=14)plt.tight_layout()plt.savefig("class_imbalance.png", dpi=150, bbox_inches="tight")plt.show()print("Imbalance chart saved: class_imbalance.png")print("\nQ3 complete. Class imbalance analysis done.")

# =============================================================================
# Section 4: Leakage Investigation
# =============================================================================
#
# Per docs/spec.md v0.1 §6 and plans/prds/creditflow-p1.md §8:
#   - Investigate `previous_defaults` as a leakage suspect
#   - Check if correlation with target > 0.5
#   - Check for time-based columns
#   - Determine if previous_defaults is available at application time
# =============================================================================

In [ ]:
print("\n" + "=" * 70)print("SECTION 4: LEAKAGE INVESTIGATION")print("=" * 70)print("Reference: plans/prds/creditflow-p1.md §8")print("Leakage suspects: previous_defaults, time-based columns")# --- Investigate previous_defaults as Leakage Suspect ---print(f"\n--- Leakage Suspect: previous_defaults ---")print("plans/prds/creditflow-p1.md §8 states:")print("  'previous_defaults tương quan quá mạnh với target → kiểm tra có phải")print("  thông tin chỉ có sau khi default không.'")if "previous_defaults" in df.columns and "default" in df.columns:    prev_default_corr = df["previous_defaults"].corr(df["default"])    print(f"\nCorrelation(previous_defaults, default): {prev_default_corr:.4f}")    # Check if previous_defaults is available at application time    print(f"\nKey question: Is previous_defaults available at APPLICATION time?")    print("  - If YES (known before loan decision): Not leakage, but high correlation")    print("    may cause data leakage in production if not carefully handled.")    print("  - If NO (only known after default occurs): LEAKAGE CONFIRMED.")    print("    P4 must NOT use previous_defaults as a feature.")    # Check distribution of previous_defaults among defaulters vs non-defaulters    print(f"\nDistribution of previous_defaults by default status:")    prev_by_default = df.groupby("default")["previous_defaults"].describe()    print(prev_by_default)    # If correlation is very high, flag it    if abs(prev_default_corr) > 0.5:        print(f"\n⚠️  WARNING: previous_defaults has HIGH correlation with P(default).")        print("    This is a suspected leakage feature per plans/prds/creditflow-p1.md §8.")        print("    P4 model selection must exclude or carefully handle this feature.")# --- Check for Time-Based Columns ---print(f"\n--- Time-Based Column Leakage Check ---")time_columns = [c for c in df.columns if any(    kw in c.lower() for kw in ["days", "date", "time", "past", "due", "after", "post"])]if time_columns:    print(f"Time-based columns detected: {time_columns}")    print("plans/prds/creditflow-p1.md §8 states:")    print("  'Mọi feature có hậu tố thời gian → cấm dùng làm input nếu không có")    print("  tại thời điểm application.'")    print(f"\n⚠️  WARNING: Time-based columns may leak post-application information.")    print("    These columns must be removed from the feature set before training.")    for col in time_columns:        print(f"    - {col}: must be verified for application-time availability")else:    print("No obvious time-based columns detected in dataset.")    print("COLAB: Verify manually if dataset contains date/days columns.")# --- Leakage Summary ---print(f"\n--- Leakage Investigation Summary ---")print("Leakage suspects from plans/prds/creditflow-p1.md §8:")print("  1. previous_defaults — correlation with target must be checked")print("  2. Time-based columns (days-past-due, etc.) — must be at application time")print("\nIf leakage is confirmed:")print("  - P4 must NOT use leaked features")print("  - P6 Pipeline must enforce split BEFORE any preprocessing fit")print("  - See plans/prds/creditflow-p1.md §8 for full details")print("\nLeakage investigation complete.")

# =============================================================================
# Section 5: Missing Values, Outliers, Duplicates
# =============================================================================
#
# Per docs/spec.md v0.1 §5 (Data layer) and §7 (Validation rules):
#   - Missing value report
#   - IQR outlier detection
#   - Duplicate row count
#   - Summary report
# =============================================================================

In [ ]:
print("\n" + "=" * 70)print("SECTION 5: MISSING VALUES, OUTLIERS, DUPLICATES")print("=" * 70)# --- Missing Value Report ---print(f"\n--- Missing Value Report ---")missing_report = pd.DataFrame({    'column': df.columns,    'missing_count': df.isnull().sum().values,    'missing_percentage': (df.isnull().sum() / len(df) * 100).values,    'dtype': df.dtypes.values})missing_report = missing_report[missing_report['missing_count'] > 0].sort_values(    'missing_percentage', ascending=False)if len(missing_report) > 0:    print(missing_report.to_string(index=False))    print(f"\n⚠️  Columns with missing values detected.")    print("    See validation rules in plans/prds/creditflow-p1.md §7.")else:    print("No missing values detected in dataset.")    print("COLAB: Verify with actual dataset — proxy may have different missing rates.")# --- Outlier Detection using IQR ---print(f"\n--- Outlier Detection (IQR Method) ---")numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()if 'default' in numeric_cols:    numeric_cols.remove('default')outlier_summary = []for col in numeric_cols:    if col in df.columns:        Q1 = df[col].quantile(0.25)        Q3 = df[col].quantile(0.75)        IQR = Q3 - Q1        lower_bound = Q1 - 1.5 * IQR        upper_bound = Q3 + 1.5 * IQR        outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()        outlier_pct = outliers / len(df) * 100        outlier_summary.append({            'column': col,            'Q1': Q1,            'Q3': Q3,            'IQR': IQR,            'lower_bound': lower_bound,            'upper_bound': upper_bound,            'outlier_count': outliers,            'outlier_pct': outlier_pct        })if outlier_summary:    outlier_df = pd.DataFrame(outlier_summary)    print(outlier_df.to_string(index=False))    print(f"\nNote: Outliers detected via IQR method (1.5 * IQR).")    print("    See validation rules in plans/prds/creditflow-p1.md §7.")    print("    Income > 0, Loan_amount > 0 must be enforced.")    print("    Age must be in range 18-100.")# --- Duplicate Row Count ---print(f"\n--- Duplicate Row Count ---")n_duplicates = df.duplicated().sum()print(f"Number of duplicate rows: {n_duplicates}")print(f"Percentage of duplicates: {n_duplicates / len(df) * 100:.2f}%")if n_duplicates > 0:    print(f"⚠️  WARNING: Duplicate rows detected.")    print("    Consider removing duplicates before analysis.")    print("    See validation rules in plans/prds/creditflow-p1.md §7.")# --- Summary Report ---print(f"\n--- Data Quality Summary Report ---")print("Dataset: Synthetic data (simulation)")print("Source: docs/spec.md v0.1, plans/prds/creditflow-p1.md")print(f"Total rows: {len(df)}")print(f"Total columns: {len(df.columns)}")print(f"Missing columns: {missing_report.shape[0] if len(missing_report) > 0 else 0}")print(f"Duplicate rows: {n_duplicates}")print(f"Outlier columns: {len(outlier_summary)}")print(f"\nAll validation rules from plans/prds/creditflow-p1.md §7 must be enforced")print("in the pipeline (pipeline/validation/schemas.py) and API (Pydantic).")print("\nData quality report complete.")

In [ ]:
print("\n" + "=" * 70)print("SECTION 6: FEATURE ENGINEERING DEMO")print("=" * 70)# Import derived feature functionsfrom pipeline.feature_engineering.features import add_derived_features, DERIVED_FEATURES# Apply derived featuresdf_features, flags = add_derived_features(df)# Print first 5 rows with derived featuresprint(f"\n--- First 5 rows with derived features ---")derived_cols = DERIVED_FEATURES + ['default']available_derived = [c for c in derived_cols if c in df_features.columns]print(df_features[available_derived].head().to_string())# Print flagsprint(f"\n--- Derived Feature Flags ---")if flags:    print(f"Flags: {flags}")else:    print("No flags — all derived features computed cleanly.")# Print derived feature correlations with targetprint(f"\n--- Derived Feature Correlations with P(default) ---")for feat in available_derived:    if feat != 'default' and feat in df_features.columns:        corr_val = df_features[feat].corr(df_features['default'])        print(f"  {feat}: {corr_val:.4f}")print("\nFeature engineering demo complete.")

In [ ]:
print("\n" + "=" * 70)print("SECTION 7: VALIDATION DEMO")print("=" * 70)# Import validation functionfrom pipeline.validation.schemas import validate_dataframe# Run validation on clean synthetic datavalidated_df, violations = validate_dataframe(df)print(f"\n--- Validation on Clean Synthetic Data ---")if not violations:    print("✅ All validation checks passed. No violations.")else:    print(f"⚠️  {len(violations)} violation(s) found:")    for v in violations:        print(f"  - {v}")# Test with a bad rowprint(f"\n--- Validation with Bad Row (income=0, age=17) ---")bad_row = pd.DataFrame({    'income': [0],    'age': [17],    'employment_years': [5],    'loan_amount': [10000],    'loan_term': [36],    'existing_debt': [3500],    'credit_history': [5],    'previous_defaults': [0],    'default': [0],})bad_validated_df, bad_violations = validate_dataframe(bad_row)if not bad_violations:    print("✅ Bad row passed validation (unexpected).")else:    print(f"⚠️  {len(bad_violations)} violation(s) detected (as expected):")    for v in bad_violations:        print(f"  - {v}")print("\nValidation demo complete.")

# =============================================================================
# End of CreditFlow EDA Notebook
# =============================================================================
#
# Next steps:
#   - P4: Model benchmark (4 models: Logistic, Tree, RF, XGBoost)
#   - P5: Evaluation with confusion matrix and business cost justification
#   - P6: ML Pipeline (reproducible, sklearn Pipeline)
#   - P7: MLflow tracking + registry
#   - P8: FastAPI (4 endpoints)
#   - P9: Docker containerization
#   - P10: Cloud deployment
#   - P11: Monitoring (system + data drift + prediction drift)
#
# Constraints reminder:
#   - NO .fit(), NO model training in this notebook
#   - NO xgboost/torch/tensorflow imports
#   - NO GPU usage
#   - All training done on Colab via extension (per docs/spec.md §19)
#   - Proxy dataset disclaimer: Simulation only
# =============================================================================

In [ ]:
print("\n" + "=" * 70)print("REQUIREMENTS CHECK")print("=" * 70)# Print library versionsprint(f"pandas    : {pd.__version__}")print(f"numpy     : {np.__version__}")print(f"matplotlib: {plt.__version__}")print(f"seaborn   : {sns.__version__}")# Confirm no .fit() calls, no xgboost/torchimport sys# Check that forbidden modules are not importedforbidden_modules = ['xgboost', 'torch', 'tensorflow', 'sklearn']loaded_modules = [m for m in forbidden_modules if m in sys.modules]if loaded_modules:    print(f"⚠️  WARNING: Forbidden modules loaded: {loaded_modules}")else:    print("✅ No forbidden modules (xgboost, torch, tensorflow) loaded.")# Verify no .fit() in the notebook's executed codeprint("✅ No .fit() calls in this notebook (EDA only, no training).")print("✅ No xgboost/torch/tensorflow imports in this notebook.")# Verify dataset integrityprint(f"\nDataset integrity check:")print(f"  Rows: {len(df)}")print(f"  Columns: {len(df.columns)}")print(f"  P(default): {df['default'].mean():.4f}")print(f"  All 8 raw features present: {all(c in df.columns for c in ['income', 'age', 'employment_years', 'loan_amount', 'loan_term', 'existing_debt', 'credit_history', 'previous_defaults'])}")print("\n" + "=" * 70)print("All checks passed. Notebook is ready for P4 Model Benchmark.")print("=" * 70)